# English → Amharic NMT — Built From Scratch (No Pretrained Weights)

This notebook trains a Transformer translation model **from random
initialization** — no `facebook/nllb-*` checkpoint, no pretrained weights,
no pretrained tokenizer. Everything (subword vocabulary, embeddings,
encoder/decoder, attention) is trained only on the parallel corpus below.

This is the **English → Amharic** direction (source = English, target =
Amharic). It's the mirror image of the Amharic → English notebook this was
adapted from — same data, same cleaning, same architecture, just source
and target swapped throughout.

**Target hardware:** Google Colab **T4 GPU** (single GPU — Colab's free-tier
default). We still pin the device defensively and confirm it before doing
any heavy work, even though Colab only exposes one GPU per session.

**Dataset:** you uploaded `amharic-english.zip` directly to the Colab
runtime and extracted it, giving the nested path
`/content/amharic-english/amharic-english` (the zip wrapped its contents in
an extra top-level folder — this is accounted for below).

**Expectations:** a from-scratch model on ~50-60k sentence pairs will not
match a fine-tuned 600M-parameter pretrained model. The goal here is a
correctly implemented, correctly trained pipeline — encoder-decoder
Transformer, custom BPE vocabulary, label smoothing, LR warmup, greedy
decoding, and BLEU evaluation on a held-out test set — not state-of-the-art
translation quality. English → Amharic is generally the harder direction
(Amharic is morphologically richer, so the decoder has a bigger space to
get right), so expect a somewhat lower BLEU than the Amharic → English run.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 0a — Unzip the dataset

Run this first if you haven't already extracted the zip this session
(Colab's local storage — including extracted files — does **not** persist
across sessions/restarts, so you'll need to re-run this each time you
reconnect, or switch to mounting Google Drive if that gets old).

In [3]:
import zipfile
import os

ZIP_PATH = "/content/amharic-english.zip"
EXTRACT_DIR = "/content/amharic-english"

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extracted to", EXTRACT_DIR)
else:
    print(f"{EXTRACT_DIR} already exists, skipping extraction")

# Quick structural check — zips often add an extra nested folder level
for root, dirs, files_ in os.walk(EXTRACT_DIR):
    depth = root.count(os.sep) - EXTRACT_DIR.count(os.sep)
    if depth <= 1:
        print(root, "->", dirs, files_[:3])


Extracted to /content/amharic-english
/content/amharic-english -> ['amharic-english'] []
/content/amharic-english/amharic-english -> ['amharic_english_JW_daily_quote', 'amharic_english_history', 'amharic_english_news', 'amharic_english_ethiopiac_bible', 'amharic_english_Jw_bible', 'amharic_english_legal'] []


## Step 0 — Pin the GPU and confirm the environment

Set `CUDA_VISIBLE_DEVICES` before importing `torch`, as good practice (even
though Colab T4 sessions are single-GPU, this avoids any risk of the "CUDA
already initialized before the env var was set" issue seen in the earlier
NLLB notebooks).

In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # must run before torch is imported anywhere
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
assert torch.cuda.device_count() == 1, (
    "More than one GPU is visible. Restart the kernel and re-run Step 0 "
    "before importing torch anywhere else."
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("Using device:", DEVICE)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


CUDA available: True
Device count: 1
Device name: Tesla T4
Using device: cuda


## Step 1 — Load and clean the parallel corpus

Same cleaning logic proven out in the earlier NLLB notebooks (Unicode NFC
normalization, Amharic homophone-character normalization, curly-quote
normalization, trailing double-period cleanup) — this part of the pipeline
has nothing to do with the model architecture, so it's reused as-is.

**Domain discovery is automatic** rather than hardcoded: rather than
assuming specific domain folder names, this scans the base directory for
any subfolder that actually contains both `amharic.txt` and `english.txt`.
Check the printed list below matches what you expect before moving on.

In [6]:
import re
import unicodedata
import pandas as pd

BASE = "/content/amharic-english/amharic-english"

# Auto-discover domain folders instead of hardcoding names — robust to
# whatever the actual folder layout turns out to be inside the zip.
domains = []
for entry in sorted(os.listdir(BASE)):
    full = os.path.join(BASE, entry)
    if os.path.isdir(full) and \
       os.path.exists(os.path.join(full, "amharic.txt")) and \
       os.path.exists(os.path.join(full, "english.txt")):
        domains.append(entry)

print(f"Discovered {len(domains)} domain folders:")
for d in domains:
    print(" -", d)
assert len(domains) > 0, f"No domain folders with amharic.txt/english.txt found under {BASE}"


Discovered 6 domain folders:
 - amharic_english_JW_daily_quote
 - amharic_english_Jw_bible
 - amharic_english_ethiopiac_bible
 - amharic_english_history
 - amharic_english_legal
 - amharic_english_news


In [7]:
def normalize_quotes(text):
    return (text.replace("\u201c", '"').replace("\u201d", '"')
                .replace("\u2018", "'").replace("\u2019", "'"))

def clean_trailing(text):
    text = text.strip()
    text = re.sub(r"\s*\.\s*\.\s*$", ".", text)
    text = re.sub(r"\.{2,}$", ".", text)
    return text

def normalize_amharic(text):
    text = unicodedata.normalize("NFC", text)
    replacements = {
        "\u1203": "\u1200", "\u1210": "\u1200", "\u1213": "\u1200",
        "\u1280": "\u1200", "\u1283": "\u1200", "\u12b8": "\u1200",
        "\u1220": "\u1230", "\u1227": "\u1236", "\u12d0": "\u12a0", "\u12d3": "\u12a0",
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    text = normalize_quotes(text)
    return clean_trailing(text)

def clean_english(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text)
    text = normalize_quotes(text)
    return clean_trailing(text)

all_pairs = []
for d in domains:
    am_path = os.path.join(BASE, d, "amharic.txt")
    en_path = os.path.join(BASE, d, "english.txt")
    with open(am_path, encoding="utf-8") as f:
        am_lines = [l.strip() for l in f.readlines()]
    with open(en_path, encoding="utf-8") as f:
        en_lines = [l.strip() for l in f.readlines()]

    n = min(len(am_lines), len(en_lines))
    for am, en in zip(am_lines[:n], en_lines[:n]):
        am_c, en_c = normalize_amharic(am), clean_english(en)
        if am_c and en_c:
            all_pairs.append({"domain": d, "am": am_c, "en": en_c})

print(f"Loaded {len(all_pairs)} raw pairs across {len(domains)} domains")


Loaded 107928 raw pairs across 6 domains


## Step 2 — Deduplicate and drop extreme length-ratio outliers

Same filtering logic as before: exact duplicate `(am, en)` pairs dropped,
and rows with a word-count ratio outside `0.15-6.0` dropped as likely
misalignments. The ratio is still computed as English words / Amharic
words (same underlying corpus stats regardless of which direction we
translate), keeping legitimate short Amharic <-> longer English pairs on
both sides.


In [8]:
df = pd.DataFrame(all_pairs).drop_duplicates(subset=["am", "en"])

df["am_wc"] = df["am"].str.split().str.len()
df["en_wc"] = df["en"].str.split().str.len()
df["len_ratio"] = df["en_wc"] / df["am_wc"].replace(0, 1)
before = len(df)
df = df[(df["len_ratio"] > 0.15) & (df["len_ratio"] < 6.0)]

print(f"After dedup + ratio filter: {len(df)} pairs (dropped {before - len(df)} outliers)")
print(df["domain"].value_counts())


After dedup + ratio filter: 64823 pairs (dropped 581 outliers)
domain
amharic_english_Jw_bible           30678
amharic_english_news               12506
amharic_english_ethiopiac_bible    11292
amharic_english_JW_daily_quote      4685
amharic_english_legal               4457
amharic_english_history             1205
Name: count, dtype: int64


## Step 3 — Train / validation / test split

Stratified by domain, same 90 / 5 / 5 split as before.

In [9]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.1, stratify=df["domain"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["domain"], random_state=SEED)

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))

os.makedirs("/content/working", exist_ok=True)
train_df.to_csv("/content/working/train.csv", index=False)
val_df.to_csv("/content/working/val.csv", index=False)
test_df.to_csv("/content/working/test.csv", index=False)
print("Saved train.csv, val.csv, test.csv to /content/working/")


Train: 58340 | Val: 3241 | Test: 3242
Saved train.csv, val.csv, test.csv to /content/working/


## Step 4 — Train subword tokenizers from scratch

No pretrained tokenizer is used. We train two **separate** SentencePiece
BPE models — one for Amharic, one for English — directly on the training
split only (never on val/test, to avoid any leakage). Separate vocabularies
make sense here since the two languages use entirely disjoint scripts
(Ge'ez vs. Latin).

Vocab size is kept modest (8,000 per language) since the training set is
only ~50-60k sentences — a much larger vocabulary would leave many subword
tokens under-trained.

### NLP preprocessing concepts, briefly

A quick glossary of the terms that come up in this tokenization step (and
a couple that don't apply here but are good to know):

- **Tokenization**: breaking raw text into smaller pieces called tokens,
  such as words, subwords, or characters. This is exactly what the
  SentencePiece BPE models below do — they learn subword tokens directly
  from the Amharic and English training text. ([GeeksforGeeks: Tokenization in NLP](https://www.geeksforgeeks.org/nlp/nlp-how-tokenizing-text-sentence-words-works/))
- **Vectorization**: converting tokens into numerical vectors so a model
  can process them — here, that's the integer token IDs produced by
  `sp.encode(..., out_type=int)`.
- **Embeddings**: a dense vector representation where tokens with similar
  meaning end up close together in a high-dimensional space. The
  `nn.Embedding` layers in the `Seq2SeqTransformer` below learn these from
  scratch during training (no pretrained embeddings are used).
- **Stemming**: chopping word endings down to a rough root (e.g. "running"
  → "run") — fast, but can produce non-words. Not used here; subword BPE
  tokenization handles morphology implicitly instead.
- **Lemmatization**: reducing a word to its proper dictionary root using
  vocabulary and morphological analysis — more linguistically correct than
  stemming but slower. Also not used here for the same reason.
- **Stop word removal**: dropping very common low-information words (e.g.
  "the", "is", "and"). Not applied in this pipeline — a translation model
  needs the full sentence, including function words, to produce fluent
  output.
- **One-hot encoding**: representing a token as an all-zero vector with a
  single 1 at its vocabulary position. This is the "before" picture that
  learned embeddings replace — one-hot vectors are sparse and carry no
  notion of similarity, which is why the embedding layer below is used
  instead.

Further reading: [An Introduction to NLP: Tokenization, Stemming, Lemmatization & More](https://medium.com/@rishichavan462/an-introduction-to-nlp-tokenization-stemming-lemmatization-more-549a22c40046) · [NLP from Tokenization to Vectorization](https://dev.to/taqiddin/nlp-from-tokenization-to-vectorization-with-practical-insights-2he1) · [Vectorization and Tokenization](https://medium.com/@rishik.8102001/vectorization-and-tokenization-unlocking-the-power-of-natural-language-processing-699ba5643c11)


In [10]:
!pip install -q sentencepiece sacrebleu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 14.7 MB/s eta 0:00:00


In [11]:
import sentencepiece as spm

SPM_DIR = "/content/working/spm"
os.makedirs(SPM_DIR, exist_ok=True)

AM_VOCAB_SIZE = 8000
EN_VOCAB_SIZE = 8000

am_train_txt = os.path.join(SPM_DIR, "am_train.txt")
en_train_txt = os.path.join(SPM_DIR, "en_train.txt")

train_df["am"].to_csv(am_train_txt, index=False, header=False)
train_df["en"].to_csv(en_train_txt, index=False, header=False)

def train_spm(input_path, model_prefix, vocab_size):
    spm.SentencePieceTrainer.train(
        input=input_path,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type="bpe",
        character_coverage=1.0,       # important for Amharic's larger character set
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        pad_piece="<pad>", unk_piece="<unk>", bos_piece="<s>", eos_piece="</s>",
    )

train_spm(am_train_txt, os.path.join(SPM_DIR, "am_spm"), AM_VOCAB_SIZE)
train_spm(en_train_txt, os.path.join(SPM_DIR, "en_spm"), EN_VOCAB_SIZE)

am_sp = spm.SentencePieceProcessor(model_file=os.path.join(SPM_DIR, "am_spm.model"))
en_sp = spm.SentencePieceProcessor(model_file=os.path.join(SPM_DIR, "en_spm.model"))

PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

print("Amharic vocab size:", am_sp.get_piece_size())
print("English vocab size:", en_sp.get_piece_size())

sample_am = train_df.iloc[0]["am"]
sample_en = train_df.iloc[0]["en"]
print("\nSample AM:", sample_am)
print("AM pieces:", am_sp.encode(sample_am, out_type=str))
print("\nSample EN:", sample_en)
print("EN pieces:", en_sp.encode(sample_en, out_type=str))


Amharic vocab size: 8000
English vocab size: 8000

Sample AM: እስከ ዛሬም ድረስ የዳጎን ካህናትና ወደ ዳጎን ቤት የሚገቡ ሁሉ በአሽዶድ የሚገኘውን የዳጎንን ደፍ የማይረግጡት ለዚህ ነው።
AM pieces: ['▁እስከ', '▁ዛሬም', '▁ድረስ', '▁የዳ', 'ጎ', 'ን', '▁ካህና', 'ትና', '▁ወደ', '▁ዳ', 'ጎ', 'ን', '▁ቤት', '▁የሚ', 'ገቡ', '▁ሁሉ', '▁በአ', 'ሽ', 'ዶ', 'ድ', '▁የሚገኘውን', '▁የዳ', 'ጎ', 'ንን', '▁ደ', 'ፍ', '▁የማይ', 'ረግ', 'ጡት', '▁ለዚህ', '▁ነው።']

Sample EN: That is why to this day, the priests of Daʹgon and all those who enter the house of Daʹgon do not walk on the threshold of Daʹgon in Ashʹdod.
EN pieces: ['▁That', '▁is', '▁why', '▁to', '▁this', '▁day', ',', '▁the', '▁priests', '▁of', '▁Da', 'ʹ', 'g', 'on', '▁and', '▁all', '▁those', '▁who', '▁enter', '▁the', '▁house', '▁of', '▁Da', 'ʹ', 'g', 'on', '▁do', '▁not', '▁walk', '▁on', '▁the', '▁thres', 'hold', '▁of', '▁Da', 'ʹ', 'g', 'on', '▁in', '▁Ash', 'ʹ', 'dod', '.']


## Step 5 — PyTorch `Dataset` / `DataLoader`

Tokenizes on the fly with the SentencePiece models trained above, wraps
each target sequence with `<s>` / `</s>`, and pads each batch dynamically
to the longest sequence in that batch (not a fixed global max length —
keeps padding waste low).

In [12]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

MAX_LEN = 128

class TranslationDataset(Dataset):
    """Source = English, target = Amharic."""
    def __init__(self, df, am_sp, en_sp, max_len=MAX_LEN):
        self.am = df["am"].tolist()
        self.en = df["en"].tolist()
        self.am_sp = am_sp
        self.en_sp = en_sp
        self.max_len = max_len

    def __len__(self):
        return len(self.en)

    def __getitem__(self, idx):
        src_ids = self.en_sp.encode(self.en[idx], out_type=int)[: self.max_len - 1] + [EOS_ID]
        tgt_ids = [BOS_ID] + self.am_sp.encode(self.am[idx], out_type=int)[: self.max_len - 2] + [EOS_ID]
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_lens = [len(s) for s in src_batch]
    tgt_lens = [len(t) for t in tgt_batch]
    src_pad = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=PAD_ID)
    tgt_pad = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_ID)
    return src_pad, tgt_pad

BATCH_SIZE = 64

train_ds = TranslationDataset(train_df, am_sp, en_sp)
val_ds = TranslationDataset(val_df, am_sp, en_sp)
test_ds = TranslationDataset(test_df, am_sp, en_sp)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Batches per epoch — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

src_sample, tgt_sample = next(iter(train_loader))
print("Sample batch shapes — src (en):", src_sample.shape, "tgt (am):", tgt_sample.shape)


Batches per epoch — train: 911, val: 51, test: 51
Sample batch shapes — src (en): torch.Size([64, 62]) tgt (am): torch.Size([64, 70])


## Step 6 — Transformer model, built from scratch

Implemented with PyTorch's `nn.TransformerEncoderLayer` /
`nn.TransformerDecoderLayer` as building blocks (so the attention math
itself is the well-tested library implementation), while the embeddings,
positional encoding, masking, encoder/decoder stacking, and output
projection are all written explicitly here rather than using `nn.Transformer`
as one opaque call.

Model size is intentionally modest given the ~50-60k sentence training set —
oversized models on this little data would just overfit.

In [13]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, nhead=8,
                 num_encoder_layers=4, num_decoder_layers=4, dim_feedforward=1024,
                 dropout=0.1, max_len=MAX_LEN):
        super().__init__()
        self.d_model = d_model

        self.src_tok_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_ID)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_decoder_layers)

        self.generator = nn.Linear(d_model, tgt_vocab_size)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src, src_key_padding_mask):
        src_emb = self.pos_enc(self.src_tok_emb(src) * math.sqrt(self.d_model))
        return self.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

    def decode(self, tgt, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask):
        tgt_emb = self.pos_enc(self.tgt_tok_emb(tgt) * math.sqrt(self.d_model))
        return self.decoder(
            tgt_emb, memory, tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )

    def forward(self, src, tgt, src_key_padding_mask, tgt_key_padding_mask):
        tgt_mask = generate_square_subsequent_mask(tgt.size(1), tgt.device)
        memory = self.encode(src, src_key_padding_mask)
        out = self.decode(tgt, memory, tgt_mask, tgt_key_padding_mask, src_key_padding_mask)
        return self.generator(out)


def generate_square_subsequent_mask(size, device):
    mask = torch.triu(torch.full((size, size), float("-inf"), device=device), diagonal=1)
    return mask

def make_padding_mask(seq, pad_id=PAD_ID):
    # True where padded (this is what PyTorch's key_padding_mask expects)
    return seq == pad_id


model = Seq2SeqTransformer(
    src_vocab_size=en_sp.get_piece_size(),   # source = English
    tgt_vocab_size=am_sp.get_piece_size(),   # target = Amharic
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,} (~{n_params/1e6:.1f}M)")


Model parameters: 13,524,800 (~13.5M)


## Step 7 — Loss, optimizer, LR warmup

Transformers trained from scratch are notoriously unstable without a
learning-rate warmup — using the Noam schedule from the original "Attention
Is All You Need" paper. Label smoothing (0.1) is standard for this kind of
translation training and tends to improve BLEU slightly.

In [14]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

D_MODEL = 256
WARMUP_STEPS = 4000

optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
# NOTE: lr=1.0 here is intentional, not a typo. LambdaLR multiplies this base
# lr by whatever noam_lr() returns each step, and noam_lr() already returns
# the full intended absolute learning rate — so the base lr must be 1.0 or
# the schedule gets silently scaled down by 1e-4x, which stalls training.

def noam_lr(step, d_model=D_MODEL, warmup_steps=WARMUP_STEPS):
    step = max(step, 1)
    return (d_model ** -0.5) * min(step ** -0.5, step * warmup_steps ** -1.5)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda s: noam_lr(s))

def run_epoch(loader, model, optimizer=None, scheduler=None, train=True):
    model.train() if train else model.eval()
    total_loss, total_tokens = 0.0, 0

    torch.set_grad_enabled(train)
    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]

        src_pad_mask = make_padding_mask(src)
        tgt_pad_mask = make_padding_mask(tgt_in)

        logits = model(src, tgt_in, src_pad_mask, tgt_pad_mask)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

        n_tokens = (tgt_out != PAD_ID).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens

    torch.set_grad_enabled(True)
    return total_loss / max(total_tokens, 1)


## Step 8 — Time a single epoch before committing to a full run

Rather than guessing total training time, measure one real epoch on this
exact GPU/model/data first, then extrapolate. This number is what actually
determines your total runtime — treat everything before this as an
estimate, and this cell's output as ground truth for your setup.

In [15]:
import time

t0 = time.time()
one_epoch_train_loss = run_epoch(train_loader, model, optimizer, scheduler, train=True)
t1 = time.time()

epoch_seconds = t1 - t0
print(f"One training epoch took {epoch_seconds:.1f}s ({epoch_seconds/60:.2f} min)")
print(f"Train loss after 1 epoch: {one_epoch_train_loss:.4f}")

N_EPOCHS_PLANNED = 30  # adjust below in Step 9 if you change this
est_total_seconds = epoch_seconds * N_EPOCHS_PLANNED
print(f"\nEstimated total time for {N_EPOCHS_PLANNED} epochs: "
      f"{est_total_seconds/3600:.2f} hours (train-only, excludes eval/decoding overhead)")


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


One training epoch took 153.4s (2.56 min)
Train loss after 1 epoch: 8.0615

Estimated total time for 30 epochs: 1.28 hours (train-only, excludes eval/decoding overhead)


## Step 9 — Full training loop

Trains for `N_EPOCHS` epochs, tracking train/val loss each epoch and saving
the checkpoint with the best validation loss. Adjust `N_EPOCHS` based on the
timing estimate from Step 8 and how much time you actually have.

In [16]:
N_EPOCHS = 30
CKPT_PATH = "/content/working/best_model.pt"

best_val_loss = float("inf")
history = {"train_loss": [one_epoch_train_loss], "val_loss": []}

# quick val pass to log against the epoch already trained in Step 8
val_loss = run_epoch(val_loader, model, train=False)
history["val_loss"].append(val_loss)
print(f"Epoch 1/{N_EPOCHS} | train_loss={one_epoch_train_loss:.4f} | val_loss={val_loss:.4f}")
if val_loss < best_val_loss:
    best_val_loss = val_loss
    torch.save(model.state_dict(), CKPT_PATH)

for epoch in range(2, N_EPOCHS + 1):
    t0 = time.time()
    train_loss = run_epoch(train_loader, model, optimizer, scheduler, train=True)
    val_loss = run_epoch(val_loader, model, train=False)
    dt = time.time() - t0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CKPT_PATH)

    print(f"Epoch {epoch}/{N_EPOCHS} | train_loss={train_loss:.4f} | "
          f"val_loss={val_loss:.4f} | {dt:.1f}s" + (" | saved (best)" if improved else ""))

print(f"\nBest val loss: {best_val_loss:.4f} — checkpoint saved to {CKPT_PATH}")


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1/30 | train_loss=8.0615 | val_loss=7.2985
Epoch 2/30 | train_loss=6.8229 | val_loss=6.2317 | 165.3s | saved (best)
Epoch 3/30 | train_loss=5.8974 | val_loss=5.5366 | 164.0s | saved (best)
Epoch 4/30 | train_loss=5.3362 | val_loss=5.1802 | 164.4s | saved (best)
Epoch 5/30 | train_loss=4.9657 | val_loss=4.8816 | 164.4s | saved (best)
Epoch 6/30 | train_loss=4.6024 | val_loss=4.6601 | 164.8s | saved (best)
Epoch 7/30 | train_loss=4.3168 | val_loss=4.4993 | 163.6s | saved (best)
Epoch 8/30 | train_loss=4.0976 | val_loss=4.3964 | 164.1s | saved (best)
Epoch 9/30 | train_loss=3.9231 | val_loss=4.3135 | 164.1s | saved (best)
Epoch 10/30 | train_loss=3.7783 | val_loss=4.2667 | 163.4s | saved (best)
Epoch 11/30 | train_loss=3.6603 | val_loss=4.2133 | 163.6s | saved (best)
Epoch 12/30 | train_loss=3.5573 | val_loss=4.1684 | 165.4s | saved (best)
Epoch 13/30 | train_loss=3.4679 | val_loss=4.1477 | 164.0s | saved (best)
Epoch 14/30 | train_loss=3.3919 | val_loss=4.1298 | 164.0s | saved (bes

## Step 10 — Greedy decoding

Loads the best checkpoint and defines a simple greedy (argmax) decoder.
Beam search would improve output quality further but greedy decoding is
the standard starting point and keeps the code easy to follow/grade.

In [17]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

def translate(sentence, max_len=MAX_LEN):
    """sentence is English; returns Amharic."""
    src_ids = en_sp.encode(sentence, out_type=int)[: max_len - 1] + [EOS_ID]
    src = torch.tensor([src_ids], dtype=torch.long, device=DEVICE)
    src_pad_mask = make_padding_mask(src)

    with torch.no_grad():
        memory = model.encode(src, src_pad_mask)
        ys = torch.tensor([[BOS_ID]], dtype=torch.long, device=DEVICE)
        for _ in range(max_len - 1):
            tgt_mask = generate_square_subsequent_mask(ys.size(1), DEVICE)
            out = model.decode(ys, memory, tgt_mask, None, src_pad_mask)
            logits = model.generator(out[:, -1, :])
            next_token = logits.argmax(dim=-1).item()
            ys = torch.cat([ys, torch.tensor([[next_token]], device=DEVICE)], dim=1)
            if next_token == EOS_ID:
                break

    out_ids = ys[0].tolist()[1:]  # drop leading <s>
    if EOS_ID in out_ids:
        out_ids = out_ids[: out_ids.index(EOS_ID)]
    return am_sp.decode(out_ids)

# sanity check on a few training examples
for i in range(3):
    en_text = train_df.iloc[i]["en"]
    am_ref = train_df.iloc[i]["am"]
    print("EN:", en_text)
    print("REF:", am_ref)
    print("MT: ", translate(en_text))
    print()


EN: That is why to this day, the priests of Daʹgon and all those who enter the house of Daʹgon do not walk on the threshold of Daʹgon in Ashʹdod.
REF: እስከ ዛሬም ድረስ የዳጎን ካህናትና ወደ ዳጎን ቤት የሚገቡ ሁሉ በአሽዶድ የሚገኘውን የዳጎንን ደፍ የማይረግጡት ለዚህ ነው።
MT:  እስከ ዛሬ ድረስ የዳጎን ካህናትና እስከ ዳጎን ቤት የሚገቡ ሁሉ በአሽዶድ የሚገኘውን የዳጎንን ደፍ በሚታገሉበት ወቅት አይሄዱም።

EN: All the families of the Shuhamites, according to those that were numbered of them, were threescore and four thousand and four hundred.
REF: የሰምዔያውያን ወገኖች ሁሉ ከእነርሱ እንደ ተቈጠሩ ስድሳ አራት ሺህ አራት መቶ ነበሩ።
MT:  የሑፋም ወገኖች ሁሉ ከእነርሱ የተቈጠሩት ስድሳ አራት ሺህ አራት መቶ ነበሩ።

EN: Five of the additional instances are found at .
REF: የአምላክ ስም የገባባቸው አምስት ተጨማሪ ቦታዎች ናቸው።
MT:  ተጨማሪ ቀልጦ የተሰራ ተጨማሪ ዋጋ ነው።



## Step 11 — BLEU evaluation on the held-out test set

Test set was never seen during training or vocabulary construction. Uses
`sacrebleu` for a standard, reproducible corpus BLEU score.

In [18]:
import sacrebleu
from tqdm.auto import tqdm

hypotheses, references = [], []
for en_text, am_text in tqdm(zip(test_df["en"].tolist(), test_df["am"].tolist()), total=len(test_df)):
    hypotheses.append(translate(en_text))
    references.append(am_text)

bleu = sacrebleu.corpus_bleu(hypotheses, [references])
print(f"Test BLEU: {bleu.score:.2f}")

for i in range(5):
    print("EN: ", test_df.iloc[i]["en"])
    print("REF:", references[i])
    print("MT: ", hypotheses[i])
    print()


  0%|          | 0/3242 [00:00<?, ?it/s]

Test BLEU: 9.54
EN:  The regional states may issue laws necessary for the implementation of this Proclamation.
REF: ክልልሎች ይህን አዋጅ ለማስፈጸም የሚያስፈልጉ ሕጎችን ሊያወጡ ይችላሉ።
MT:  ክልሉ መንግሥት ይህን አዋጅ ለማስፈፀም የሚያስፈልግ ደንብ ሊያወጣ ይችላል፡፡

EN:  Who set its measurements, in case you know,Or who stretched a measuring line across it?
REF: የምታውቅ ከሆነ፣ መለኪያዎቿን የወሰነ፣ወይስ መለኪያ ገመድ በላይዋ ላይ የዘረጋ ማን ነው?
MT:  የተለየው እርምጃ የተዘረጋው ሀሳብ ማን ነው?ወይስ የመለኪያ ገመድ የተዘረጋለት ማን ነው?

EN:  Now Mirʹi·am and Aaron began to speak against Moses because of the Cushʹite wife he had married, for he had taken a Cushʹite wife.
REF: ሙሴ ኢትዮጵያዊት ሴት አግብቶ ስለነበር ባገባት ኢትዮጵያዊት ሴት የተነሳ ሚርያምና አሮን ይነቅፉት ጀመር።
MT:  በዚህ ጊዜ ሚርያምና አሮን ከሉሻሽ ሚስት ጋር ስለወሰደ ሙሴ፣ የሙሻዊ ሚስትም ነበረቻቸው።

EN:  But Cook says establishing ties with North Korea doesn't mean London will turn a blind eye to human rights abuses in the country.
REF: ግን ኩክ እንደተናገሩት ከሰሜን ኮሪያ ጋር ግንኙነት መመሥረተ ማለት በዚች አገር ውስጥ ለሚደርሰው የሰብአዊ መብት መጣስ ለንደን አይኗን ጨፍና ትቀመጣች ማለት አይደለም።
MT:  ግን ሰሜን ኮሪያ የኒውክሌር ጦር መሣሪያ ለማይላት እንዷለን።

EN:

## Step 12 — Save final artifacts

Saves the trained model weights and both SentencePiece tokenizers so the
model can be reloaded later without retraining.

In [19]:
import shutil

FINAL_DIR = "/content/working/final_model"
os.makedirs(FINAL_DIR, exist_ok=True)

torch.save(model.state_dict(), os.path.join(FINAL_DIR, "model.pt"))
shutil.copy(os.path.join(SPM_DIR, "am_spm.model"), os.path.join(FINAL_DIR, "am_spm.model"))
shutil.copy(os.path.join(SPM_DIR, "en_spm.model"), os.path.join(FINAL_DIR, "en_spm.model"))

import json
with open(os.path.join(FINAL_DIR, "config.json"), "w") as f:
    json.dump({
        "d_model": D_MODEL,
        "direction": "en->am",
        "src_vocab_size": en_sp.get_piece_size(),
        "tgt_vocab_size": am_sp.get_piece_size(),
        "max_len": MAX_LEN,
        "test_bleu": bleu.score,
    }, f, indent=2)

print("Saved final model + tokenizers + config to", FINAL_DIR)


Saved final model + tokenizers + config to /content/working/final_model
